In [1]:
!pip install pandas numpy plotly pytz

In [2]:
#importing libraries
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz

In [3]:
#loading dataset
df = pd.read_csv("C:/Users/yashr/Desktop/eve2/archive (6)/google_play_store_dataset.csv")

df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [4]:
print(df.columns)

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='object')


In [5]:
#cleaning installed columns
df['Installs'] = (
    df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

df['Installs'] = pd.to_numeric(
    df['Installs'],
    errors='coerce'
)

df = df.dropna(subset=['Installs'])

df['Installs'] = df['Installs'].astype(int)

df[['Category', 'Installs']].head()

,Category,Installs
0,ART_AND_DESIGN,10000
1,ART_AND_DESIGN,500000
2,ART_AND_DESIGN,5000000
3,ART_AND_DESIGN,50000000
4,ART_AND_DESIGN,100000


In [6]:
#Removing Categories Starting With A, C, G, S
excluded_prefixes = ('A', 'C', 'G', 'S')

filtered_df = df[
    ~df['Category'].str.startswith(excluded_prefixes)
]

print(filtered_df['Category'].unique())

['BEAUTY' 'BOOKS_AND_REFERENCE' 'BUSINESS' 'DATING' 'EDUCATION'
 'ENTERTAINMENT' 'EVENTS' 'FINANCE' 'FOOD_AND_DRINK' 'HEALTH_AND_FITNESS'
 'HOUSE_AND_HOME' 'LIBRARIES_AND_DEMO' 'LIFESTYLE' 'FAMILY' 'MEDICAL'
 'PHOTOGRAPHY' 'TRAVEL_AND_LOCAL' 'TOOLS' 'PERSONALIZATION' 'PRODUCTIVITY'
 'PARENTING' 'WEATHER' 'VIDEO_PLAYERS' 'NEWS_AND_MAGAZINES'
 'MAPS_AND_NAVIGATION']


In [7]:
#finding top 5 categories 
top_categories = (
    filtered_df
    .groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

top_categories

Category
PRODUCTIVITY          14176091369
TOOLS                 11452771915
FAMILY                10258263505
PHOTOGRAPHY           10088247655
NEWS_AND_MAGAZINES     7496317760
Name: Installs, dtype: int64

In [8]:
top5_list = top_categories.index.tolist()

filtered_df = filtered_df[
    filtered_df['Category'].isin(top5_list)
]

filtered_df['Category'].value_counts()

Category
FAMILY                1972
TOOLS                  843
PRODUCTIVITY           424
PHOTOGRAPHY            335
NEWS_AND_MAGAZINES     283
Name: count, dtype: int64

In [9]:
#creating country column
countries = [
    'India',
    'United States',
    'Brazil',
    'Germany',
    'United Kingdom',
    'Canada',
    'Australia',
    'France',
    'Japan',
    'South Africa'
]

np.random.seed(42)

filtered_df['Country'] = np.random.choice(
    countries,
    size=len(filtered_df)
)

filtered_df[['Country', 'Category']].head()

,Country,Category
2014,Australia,FAMILY
2015,Germany,FAMILY
2016,France,FAMILY
2017,United Kingdom,FAMILY
2018,Australia,FAMILY


In [10]:
#aggregating data
country_df = (
    filtered_df
    .groupby(['Country', 'Category'])
    ['Installs']
    .sum()
    .reset_index()
)

country_df.head()

,Country,Category,Installs
0,Australia,FAMILY,661224945
1,Australia,NEWS_AND_MAGAZINES,552117010
2,Australia,PHOTOGRAPHY,1535163100
3,Australia,PRODUCTIVITY,218469891
4,Australia,TOOLS,1472098425


In [11]:
#create highlight column
country_df['Highlight'] = np.where(
    country_df['Installs'] > 1_000_000,
    'Above 1M',
    'Below 1M'
)

country_df.head()

,Country,Category,Installs,Highlight
0,Australia,FAMILY,661224945,Above 1M
1,Australia,NEWS_AND_MAGAZINES,552117010,Above 1M
2,Australia,PHOTOGRAPHY,1535163100,Above 1M
3,Australia,PRODUCTIVITY,218469891,Above 1M
4,Australia,TOOLS,1472098425,Above 1M


In [12]:
#checking current IST time 
ist = pytz.timezone('Asia/Kolkata')

current_time = datetime.now(ist)

print(current_time)

2026-06-23 11:32:01.867016+05:30


In [13]:
#time restriction 
current_hour = current_time.hour

show_graph = (
    current_hour >= 18 and
    current_hour < 20
)

print(show_graph)

False


In [15]:
#Creating  Choropleth Map
fig = px.choropleth(
    country_df,
    locations='Country',
    locationmode='country names',
    color='Installs',
    hover_name='Country',
    hover_data=[
        'Category',
        'Installs',
        'Highlight'
    ],
    animation_frame='Category',
    color_continuous_scale='Viridis',
    title='Global Installs by Top 5 Categories'
)

fig.update_layout(
    height=600,
    margin=dict(
        l=20,
        r=20,
        t=60,
        b=20
    )
)

C:\Users\yashr\AppData\Local\Temp\ipykernel_25784\4164225497.py:2: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [ ]:
#visualization is available only between 6 PM and 8 PM IST
if show_graph:
    fig.show()
else:
    print(
        "Dashboard visualization is available only between 6 PM and 8 PM IST."
    )

Dashboard visualization is available only between 6 PM and 8 PM IST.


In [16]:
fig = px.choropleth(
    country_df,
    locations='Country',
    locationmode='country names',
    color='Installs',
    animation_frame='Category',
    hover_name='Country',
    hover_data=[
        'Installs',
        'Highlight'
    ],
    color_continuous_scale='Viridis',
    title='Global Installs by Top 5 Categories'
)

high_install = country_df[
    country_df['Installs'] > 1_000_000
]

fig.add_scattergeo(
    locations=high_install['Country'],
    locationmode='country names',
    mode='markers',
    marker=dict(size=10),
    text=high_install['Country'],
    name='Above 1M Installs'
)

fig.update_layout(height=700)

C:\Users\yashr\AppData\Local\Temp\ipykernel_28088\1722400405.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(
